# **Debt-Adjusted Value (EBITDA / Enterprise Value)**

This study asks whether, after Islamic business and leverage screens, ranking Halal stocks by **EBITDA / Enterprise Value** (cheap EV) beats ranking by **earnings yield (E/P)**, and whether AAOIFI ratio screens help avoid classic **value traps**.

After dropping banned businesses and names above the AAOIFI debt cap (`< 30%`), we:

1. Require **positive TTM EBITDA** and a usable **enterprise value** (`EV > 0`)
2. Rank by **EBITDA/EV** (high = cheap) and own the top slice **in proportion to company size**

**Enterprise value:** \(\mathrm{EV} = \mathrm{Market\ Cap} + \mathrm{Total\ Debt} - \mathrm{Cash}\)

**White-paper focus:** preventing value traps by enforcing point-in-time AAOIFI financial-ratio screens on historically cheap stocks.

**Control twin:** same Halal filters, ranked by **earnings yield** (`Net Income / Market Cap`) instead of EBITDA/EV.


## Setup

Install dependencies and configure the backtest window.

`FAST_MODE = True` uses a 60-name smoke-test universe. Set it to `False` for the full S&P 500 paper run. Performance stats start on the first day the strategy actually holds stocks, not on `START`.

**SEC fundamentals:** AAOIFI screens need SEC EDGAR history. The setup cell installs the local `halalquant` repo (`Development/halalquant`) when present, otherwise PyPI. Set `HALALQUANT_SEC_UA` to your name and email if SEC requests fail. **Restart the kernel** after the first setup run so imports pick up the updated package.


In [ ]:
import os
import subprocess
import sys
from datetime import date
from io import StringIO
from pathlib import Path

import requests
from IPython.display import display


def resolve_notebook_dir() -> Path:
    """Jupyter sometimes loses cwd(); find this notebook's folder reliably."""
    candidates = [Path.cwd()]
    candidates.append(
        Path.home() / "Documents/Development/Monterey-Finance/Research/papers/08-debt-adj-value"
    )
    for path in candidates:
        try:
            resolved = path.resolve()
        except OSError:
            continue
        if resolved.is_dir() and (resolved / "code.ipynb").exists():
            return resolved
    return Path.cwd()


NB_DIR = resolve_notebook_dir()
os.chdir(NB_DIR)

HQ_ROOT = NB_DIR.parents[3] / "halalquant"
PIP_DEPS = ["yfinance", "matplotlib", "requests", "lxml", "html5lib"]

if HQ_ROOT.is_dir() and (HQ_ROOT / "pyproject.toml").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(HQ_ROOT)],
        cwd=str(NB_DIR),
        check=False,
    )
    hq_src = str(HQ_ROOT)
    if hq_src not in sys.path:
        sys.path.insert(0, hq_src)
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "halalquant"] + PIP_DEPS,
        cwd=str(NB_DIR),
        check=False,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + PIP_DEPS,
    cwd=str(NB_DIR),
    check=False,
)

os.environ.setdefault(
    "HALALQUANT_SEC_UA",
    "Monterey Finance Research halalquant/0.1.0 research@montereyfinance.com",
)

import warnings

import halalquant as hq
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from halalquant.providers._yfinance import YFinanceProvider
from halalquant.screening._aaoifi import AAOIFIScreener
from halalquant.screening._sector_filter import SectorFilter

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Backtest configuration ---
START = "2020-01-01"
END = "2025-12-31"
KEEP_QUANTILE = 0.20          # top quintile by EBITDA/EV (or earnings yield)
MIN_HOLDINGS = 8              # skip a rebalance if the screen is too thin
REBALANCE_FREQ = "ME"         # "ME" = monthly, "QE" = quarterly
DEBT_THRESHOLD = 0.30         # AAOIFI debt / 24m market cap limit
FILING_LAG_DAYS = 90          # conservative PIT lag for yfinance statement dates
FAST_MODE = True              # True → 60 names (smoke test); False → full S&P 500 for the paper
RISK_FREE_RATE = 0.02
COST_BPS = 10
STRATEGY_LABEL = "EBITDA/EV value"
PE_LABEL = "E/P twin"         # same Halal filters, ranked by earnings yield

BENCHMARKS = {
    "SPY": "SPY",
    "S&P 500": "^GSPC",
    "SPUS": "SPUS",
}

print(f"notebook dir: {NB_DIR}")
print(f"halalquant v{hq.__version__} from {Path(hq.__file__).resolve().parent.parent}")
print(f"Window: {START} → {END}  FAST_MODE={FAST_MODE}  MIN_HOLDINGS={MIN_HOLDINGS}")


## Step 1 — Start with a universe of stocks

We use **S&P 500 constituents** as the broad US equity universe, then pre-filter excluded business activities (banks, alcohol, gambling, etc.) using `halalquant`'s sector screen.


In [ ]:
FALLBACK_SP500 = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "BRK-B", "LLY", "AVGO", "JPM",
    "UNH", "XOM", "V", "MA", "PG", "COST", "HD", "JNJ", "ABBV", "NFLX",
    "CRM", "MRK", "AMD", "PEP", "KO", "TMO", "ADBE", "WMT", "CSCO", "ACN",
    "MCD", "LIN", "ABT", "DHR", "INTC", "CMCSA", "TXN", "QCOM", "INTU", "AMAT",
    "DIS", "IBM", "GE", "CAT", "NOW", "VZ", "AMGN", "PFE", "ISRG", "GS",
    "PM", "T", "MO", "CVX", "COP", "NEE", "BMY", "MDT", "UPS", "HON",
]


def load_sp500_tickers() -> list[str]:
    """Fetch current S&P 500 symbols from Wikipedia (with offline fallback)."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "MontereyFinanceResearch/1.0 (halal-quant-notebook)"}
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        table = pd.read_html(StringIO(response.text), attrs={"id": "constituents"})[0]
        return (
            table["Symbol"]
            .astype(str)
            .str.replace(".", "-", regex=False)
            .tolist()
        )
    except Exception as exc:
        print(f"Could not fetch S&P 500 list ({exc}); using fallback basket.")
        return FALLBACK_SP500.copy()


def apply_sector_screen(tickers: list[str]) -> tuple[list[str], pd.DataFrame, dict[str, str]]:
    """Remove non-compliant sectors before financial ratio screens run."""
    provider = YFinanceProvider()
    sector_map = provider.get_sector_map(tickers)
    sector_filter = SectorFilter()
    kept = sector_filter.filter_symbols(tickers, sector_map=sector_map)
    audit = pd.DataFrame(sector_filter.audit_log)
    return kept, audit, sector_map


raw_universe = load_sp500_tickers()
if FAST_MODE:
    raw_universe = raw_universe[:60]
    print("FAST_MODE is on — smoke-test universe only. Set FAST_MODE = False for the paper.\n")

eligible_universe, sector_audit, activity_labels = apply_sector_screen(raw_universe)

print(f"Raw universe:      {len(raw_universe)} tickers")
print(f"After sector screen: {len(eligible_universe)} tickers")
print(f"Removed by sector:   {len(raw_universe) - len(eligible_universe)}")

sector_audit.head(10)


## Steps 2–6 — Filter, score EBITDA/EV, select, rebalance

Pipeline at each rebalance date (point-in-time, no look-ahead):

1. **FILTER** — AAOIFI screens (`debt < 30%`, cash & receivables caps)
2. **SCORE** — attach lagged TTM EBITDA and net income; compute
   - \(\mathrm{EV} = \mathrm{Market\ Cap} + \mathrm{Total\ Debt} - \mathrm{Cash}\)
   - \(\mathrm{EBITDA/EV}\) (live signal)
   - \(\mathrm{E/P} = \mathrm{Net\ Income} / \mathrm{Market\ Cap}\) (twin)
3. **SELECT** — keep positive EBITDA, \(\mathrm{EV} > 0\), positive net income for the twin path; take the top `KEEP_QUANTILE` by the ranking column
4. **WEIGHT** — cap-weight survivors
5. **REBALANCE** — monthly (`ME`); fail AAOIFI or drop out of the top slice → exit next month-end

A **P/E twin** uses the same Halal filters, then ranks by earnings yield instead of EBITDA/EV — so any edge is the debt-adjusted multiple, not a different compliance rule.


In [ ]:
def _yahoo_row(frame: pd.DataFrame, labels: tuple[str, ...]) -> pd.Series | None:
    """Return the first matching income-statement row (case-insensitive)."""
    if frame is None or frame.empty:
        return None
    index_map = {str(i).strip().lower(): i for i in frame.index}
    for label in labels:
        key = label.strip().lower()
        if key in index_map:
            return frame.loc[index_map[key]]
    return None


def build_income_history(symbols: list[str], filing_lag_days: int = FILING_LAG_DAYS) -> pd.DataFrame:
    """Annual EBITDA and net income with a conservative public filing lag."""
    ebitda_labels = ("EBITDA", "Ebitda")
    net_income_labels = (
        "Net Income",
        "Net Income Common Stockholders",
        "Net Income Continuous Operations",
    )
    op_income_labels = ("Operating Income", "EBIT")
    da_labels = (
        "Reconciled Depreciation",
        "Depreciation And Amortization",
        "Depreciation",
    )

    rows: list[dict] = []
    n_symbols = len(symbols)
    for i, symbol in enumerate(symbols, 1):
        if i == 1 or i % 50 == 0 or i == n_symbols:
            print(f"  Income history {i}/{n_symbols}", flush=True)
        try:
            financials = yf.Ticker(symbol).financials
        except Exception:
            continue
        if financials is None or financials.empty:
            continue

        ebitda_row = _yahoo_row(financials, ebitda_labels)
        ni_row = _yahoo_row(financials, net_income_labels)
        op_row = _yahoo_row(financials, op_income_labels)
        da_row = _yahoo_row(financials, da_labels)

        for period_end in financials.columns:
            ebitda = np.nan
            if ebitda_row is not None:
                ebitda = ebitda_row.at[period_end]
            elif op_row is not None:
                op = op_row.at[period_end]
                da = da_row.at[period_end] if da_row is not None else np.nan
                if pd.notna(op) and pd.notna(da):
                    ebitda = float(op) + float(da)  # D&A often stored positive
                elif pd.notna(op):
                    ebitda = float(op)

            net_income = ni_row.at[period_end] if ni_row is not None else np.nan
            if pd.isna(ebitda) and pd.isna(net_income):
                continue

            report_ts = pd.Timestamp(period_end).normalize()
            filed_ts = report_ts + pd.Timedelta(days=filing_lag_days)
            rows.append(
                {
                    "symbol": symbol,
                    "report_date": report_ts.date(),
                    "filed_date": filed_ts.date(),
                    "ebitda": float(ebitda) if pd.notna(ebitda) else np.nan,
                    "net_income": float(net_income) if pd.notna(net_income) else np.nan,
                }
            )

    return pd.DataFrame(rows)


def attach_latest_income(metrics: pd.DataFrame, income_history: pd.DataFrame) -> pd.DataFrame:
    """Join the latest income filing known on each metrics snapshot date."""
    if metrics.empty:
        out = metrics.copy()
        out["ebitda"] = np.nan
        out["net_income"] = np.nan
        return out
    if income_history.empty:
        out = metrics.copy()
        out["ebitda"] = np.nan
        out["net_income"] = np.nan
        return out

    hist = income_history.copy()
    hist["filed_date"] = pd.to_datetime(hist["filed_date"])
    hist["report_date"] = pd.to_datetime(hist["report_date"])

    pieces: list[pd.DataFrame] = []
    for as_of, snap in metrics.groupby("as_of", sort=True):
        as_of_ts = pd.Timestamp(as_of)
        snap = snap.copy()
        ebitdas: list[float] = []
        net_incomes: list[float] = []
        for _, row in snap.iterrows():
            known = hist[(hist["symbol"] == row["symbol"]) & (hist["filed_date"] <= as_of_ts)]
            if known.empty:
                ebitdas.append(np.nan)
                net_incomes.append(np.nan)
            else:
                latest = known.sort_values("report_date").iloc[-1]
                ebitdas.append(float(latest["ebitda"]) if pd.notna(latest["ebitda"]) else np.nan)
                net_incomes.append(float(latest["net_income"]) if pd.notna(latest["net_income"]) else np.nan)
        snap["ebitda"] = ebitdas
        snap["net_income"] = net_incomes
        pieces.append(snap)
    return pd.concat(pieces, ignore_index=True)


def price_on_or_before(price_panel: pd.DataFrame, symbol: str, as_of) -> float:
    if symbol not in price_panel.columns:
        return np.nan
    s = price_panel[symbol].dropna().loc[: pd.Timestamp(as_of)]
    return float(s.iloc[-1]) if not s.empty else np.nan


def assign_cap_weights(selected: pd.DataFrame) -> pd.DataFrame:
    """Bigger firms get more of the book. Equal-weight if market cap is unusable."""
    out = selected.copy()
    if out.empty:
        out["weight"] = pd.Series(dtype=float)
        return out
    cap = pd.to_numeric(out["market_cap"], errors="coerce").clip(lower=0)
    total = float(cap.sum(skipna=True))
    if np.isfinite(total) and total > 0:
        out["weight"] = cap / total
        out["weight"] = out["weight"].fillna(0.0)
        wsum = float(out["weight"].sum())
        out["weight"] = out["weight"] / wsum if wsum > 0 else 1.0 / len(out)
    else:
        out["weight"] = 1.0 / len(out)
    return out


def score_debt_adjusted_value(panel: pd.DataFrame) -> pd.DataFrame:
    """Attach EV, EBITDA/EV, and earnings yield on an already-filtered snapshot."""
    out = panel.copy()
    if out.empty:
        return out

    mcap = pd.to_numeric(out.get("market_cap"), errors="coerce")
    debt = pd.to_numeric(out.get("total_debt"), errors="coerce").fillna(0.0)
    cash = pd.to_numeric(out.get("cash_and_equiv"), errors="coerce").fillna(0.0)
    ebitda = pd.to_numeric(out.get("ebitda"), errors="coerce")
    net_income = pd.to_numeric(out.get("net_income"), errors="coerce")

    out["market_cap"] = mcap
    out["total_debt"] = debt
    out["cash_and_equiv"] = cash
    out["ebitda"] = ebitda
    out["net_income"] = net_income
    out["enterprise_value"] = mcap + debt - cash
    out["ebitda_yield"] = ebitda / out["enterprise_value"].replace(0, np.nan)
    out["earnings_yield"] = net_income / mcap.replace(0, np.nan)
    out["ev_ebitda"] = out["enterprise_value"] / ebitda.replace(0, np.nan)
    return out


def select_debt_adjusted_value(
    panel: pd.DataFrame,
    as_of: date,
    rank_col: str = "ebitda_yield",
    keep_quantile: float = KEEP_QUANTILE,
    min_holdings: int = MIN_HOLDINGS,
    screener: AAOIFIScreener | None = None,
) -> pd.DataFrame:
    """FILTER (Halal) → SCORE → positive EBITDA/EV (or E/P) → top quantile → cap-weight."""
    empty_cols = [
        "symbol", "ebitda_yield", "earnings_yield", "ev_ebitda",
        "enterprise_value", "ebitda", "net_income", "market_cap", "weight",
        "total_debt", "cash_and_equiv", "debt_ratio", "cash_ratio", "receivables_ratio",
    ]
    screener = screener or AAOIFIScreener(debt_threshold=DEBT_THRESHOLD)
    as_of_d = pd.Timestamp(as_of).date()
    snap = panel[pd.to_datetime(panel["as_of"]).dt.date == as_of_d].copy()
    if snap.empty:
        return pd.DataFrame(columns=empty_cols)

    screened = screener.evaluate_compliance(snap)
    snap = snap.drop(columns=[c for c in ("debt_ratio", "cash_ratio", "receivables_ratio") if c in snap.columns])
    passed = snap.merge(screened, on="symbol", how="inner")
    passed = passed[passed["is_compliant"].fillna(False)].copy()
    passed = score_debt_adjusted_value(passed)

    passed = passed[
        pd.to_numeric(passed["market_cap"], errors="coerce").gt(0)
        & pd.to_numeric(passed["enterprise_value"], errors="coerce").gt(0)
        & pd.to_numeric(passed["ebitda"], errors="coerce").gt(0)
        & passed["ebitda_yield"].notna()
        & np.isfinite(passed["ebitda_yield"])
        & passed["ebitda_yield"].gt(0)
    ].copy()

    # Earnings-yield twin still needs positive NI; live book keeps NI as a diagnostic only
    if rank_col == "earnings_yield":
        passed = passed[
            passed["earnings_yield"].notna()
            & np.isfinite(passed["earnings_yield"])
            & passed["earnings_yield"].gt(0)
        ].copy()

    if passed.empty:
        return pd.DataFrame(columns=empty_cols)

    effective_min = min(min_holdings, max(3, len(passed)))
    if len(passed) < 3:
        return pd.DataFrame(columns=empty_cols)

    cutoff = passed[rank_col].quantile(1.0 - keep_quantile)
    selected = passed[passed[rank_col] >= cutoff].copy()
    if len(selected) < effective_min:
        selected = passed.nlargest(effective_min, rank_col).copy()

    selected = assign_cap_weights(selected)
    keep = [c for c in empty_cols if c in selected.columns]
    return selected[keep].sort_values("weight", ascending=False)


def run_debt_adjusted_value_backtest(
    metrics: pd.DataFrame,
    income_history: pd.DataFrame,
    prices: pd.DataFrame,
    rank_col: str = "ebitda_yield",
    keep_quantile: float = KEEP_QUANTILE,
    min_holdings: int = MIN_HOLDINGS,
) -> tuple[pd.DataFrame, pd.DataFrame, list[date]]:
    """Cap-weighted Halal debt-adjusted value book. Pre-live days are missing, not 0%."""
    panel = attach_latest_income(metrics, income_history)
    screener = AAOIFIScreener(debt_threshold=DEBT_THRESHOLD)

    price_panel = (
        prices.pivot(index="date", columns="symbol", values="adj_close")
        .sort_index()
        .ffill()
    )
    price_panel.index = pd.to_datetime(price_panel.index)
    daily_returns = price_panel.pct_change(fill_method=None)

    rebalance_dates = sorted(pd.to_datetime(panel["as_of"].dropna().unique()).date)
    weights_by_date: dict[date, pd.Series] = {}
    selection_log: list[pd.DataFrame] = []

    for as_of in rebalance_dates:
        picks = select_debt_adjusted_value(
            panel,
            as_of=as_of,
            rank_col=rank_col,
            keep_quantile=keep_quantile,
            min_holdings=min_holdings,
            screener=screener,
        )
        if picks.empty:
            weights_by_date[as_of] = pd.Series(dtype=float)
        else:
            weights_by_date[as_of] = picks.set_index("symbol")["weight"].astype(float)
            selection_log.append(picks.assign(as_of=as_of))

    selection_df = (
        pd.concat(selection_log, ignore_index=True)
        if selection_log
        else pd.DataFrame(columns=["symbol", "ebitda_yield", "earnings_yield", "weight", "as_of"])
    )

    port_rows: list[dict] = []
    active_w = pd.Series(dtype=float)
    last_rebalance: date | None = None
    live = False

    for dt in daily_returns.index:
        if dt.date() < pd.Timestamp(START).date():
            continue

        prior = [d for d in rebalance_dates if pd.Timestamp(d) <= dt]
        if prior and prior[-1] != last_rebalance:
            active_w = weights_by_date.get(prior[-1], pd.Series(dtype=float))
            last_rebalance = prior[-1]
            if not active_w.empty:
                live = True

        if not live:
            port_rows.append({"date": dt.date(), "return": np.nan, "n_holdings": 0})
            continue

        if active_w.empty:
            port_rows.append({"date": dt.date(), "return": 0.0, "n_holdings": 0})
            continue

        held = [s for s in active_w.index if s in daily_returns.columns]
        w = active_w.reindex(held).astype(float)
        w = w[w > 0]
        if w.empty:
            port_rows.append({"date": dt.date(), "return": 0.0, "n_holdings": 0})
            continue
        w = w / w.sum()
        day_ret = (daily_returns.loc[dt, w.index] * w).sum(skipna=True)
        port_rows.append({
            "date": dt.date(),
            "return": 0.0 if pd.isna(day_ret) else float(day_ret),
            "n_holdings": int(len(w)),
        })

    port_returns = pd.DataFrame(port_rows)
    port_returns["date"] = pd.to_datetime(port_returns["date"])
    return port_returns, selection_df, rebalance_dates


def calc_performance_stats(daily_returns: pd.Series, label: str, rf: float = RISK_FREE_RATE) -> dict:
    """CAGR, volatility, Sharpe, Sortino, max drawdown, Calmar."""
    r = daily_returns.dropna()
    if r.empty:
        return {
            "Strategy": label, "CAGR": np.nan, "Volatility": np.nan, "Sharpe": np.nan,
            "Sortino": np.nan, "Max Drawdown": np.nan, "Calmar": np.nan,
        }

    equity = (1 + r).cumprod()
    years = len(r) / 252
    cagr = equity.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    vol = r.std() * np.sqrt(252)
    excess = r - rf / 252
    sharpe = excess.mean() / excess.std() * np.sqrt(252) if excess.std() > 0 else np.nan
    downside = r[r < 0]
    sortino = (r.mean() - rf / 252) / downside.std() * np.sqrt(252) if len(downside) else np.nan
    dd = equity / equity.cummax() - 1
    max_dd = dd.min()
    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

    return {
        "Strategy": label,
        "CAGR": cagr,
        "Volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown": max_dd,
        "Calmar": calmar,
    }


print("Helpers ready: income history / EV / EBITDA yield / E/P twin / backtest")


### Download prices, fundamentals, and income statements

- Pulls annual Yahoo income statements (EBITDA + net income) with a 90-day filing lag
- Pulls point-in-time AAOIFI financial snapshots (includes `total_debt`, `cash_and_equiv`, `market_cap`)
- Downloads adjusted prices for the eligible universe and benchmarks


In [ ]:
print("Fetching income history (EBITDA + net income) …")
income_history = build_income_history(eligible_universe)
print(
    f"  {income_history['symbol'].nunique() if not income_history.empty else 0} symbols with income data  ·  "
    f"{len(income_history)} annual rows"
)

print("Fetching point-in-time AAOIFI metrics …")
metrics = hq.get_financial_metrics(
    eligible_universe,
    start=START,
    end=END,
    freq=REBALANCE_FREQ,
)
print(f"  {metrics.shape[0]} snapshot rows")

# halalquant.validate_symbols rejects index tickers like ^GSPC — fetch those via yfinance.
hq_ok = {t for t in set(BENCHMARKS.values()) if t.replace(".", "").replace("-", "").isalnum()}
index_tickers = sorted(set(BENCHMARKS.values()) - hq_ok)
all_symbols = sorted(set(eligible_universe) | hq_ok)

print("Downloading price history …")
prices = hq.download(all_symbols, start=START, end=END)
prices["date"] = pd.to_datetime(prices["date"])

if index_tickers:
    print(f"  Fetching index benchmarks via yfinance: {index_tickers}")
    raw = yf.download(
        index_tickers,
        start=START,
        end=END,
        auto_adjust=False,
        progress=False,
        group_by="ticker",
        threads=False,
    )
    index_frames = []
    for ticker in index_tickers:
        if isinstance(raw.columns, pd.MultiIndex):
            if ticker not in raw.columns.get_level_values(0):
                print(f"  Warning: no data for {ticker}")
                continue
            sub = raw[ticker].copy()
        else:
            sub = raw.copy()
        sub = sub.rename(columns=str.title)
        adj = sub["Adj Close"] if "Adj Close" in sub.columns else sub["Close"]
        dates = pd.to_datetime(sub.index)
        if getattr(dates, "tz", None) is not None:
            dates = dates.tz_convert("UTC").tz_localize(None)
        frame = pd.DataFrame({
            "symbol": ticker,
            "date": dates,
            "open": sub.get("Open"),
            "high": sub.get("High"),
            "low": sub.get("Low"),
            "close": sub.get("Close"),
            "volume": sub.get("Volume"),
            "adj_close": adj,
        })
        index_frames.append(frame.dropna(subset=["adj_close"]))
    if index_frames:
        prices = pd.concat([prices, *index_frames], ignore_index=True)

print(f"  {prices['symbol'].nunique()} symbols, {prices['date'].nunique()} trading days")

needed = ["market_cap", "total_debt", "cash_and_equiv"]
missing = [c for c in needed if c not in metrics.columns]
if missing:
    print(f"WARNING: metrics panel missing {missing} — EV cannot be scored.")
else:
    cov = metrics[needed].notna().mean()
    print("Fundamentals coverage:")
    for col, rate in cov.items():
        print(f"  {col}: {rate:.1%}")

metrics.head()


### Run the backtest

Sanity-checks the last rebalance funnel, then runs the live **EBITDA/EV** book and the **E/P twin**.


In [ ]:
# Funnel on the last rebalance date
_as_of = sorted(pd.to_datetime(metrics["as_of"].dropna().unique()).date)[-1]
_panel = attach_latest_income(metrics, income_history)
_screener = AAOIFIScreener(debt_threshold=DEBT_THRESHOLD)
_snap = _panel[pd.to_datetime(_panel["as_of"]).dt.date == _as_of]
_halal = _snap.merge(_screener.evaluate_compliance(_snap), on="symbol", how="inner")
_halal = _halal[_halal["is_compliant"].fillna(False)]
_scored = select_debt_adjusted_value(_panel, _as_of)
print(
    f"Funnel on {_as_of}: fundamentals={len(_snap)}  Halal={len(_halal)}  "
    f"EBITDA/EV picks={len(_scored)}  MIN_HOLDINGS={MIN_HOLDINGS}"
)
if not _scored.empty:
    print(
        f"EBITDA/EV range: {_scored['ebitda_yield'].min():.2%} → {_scored['ebitda_yield'].max():.2%}  "
        f"(median {_scored['ebitda_yield'].median():.2%})  ·  "
        f"median EV/EBITDA {_scored['ev_ebitda'].median():.1f}x"
    )

strategy_returns, selections, rebalance_dates = run_debt_adjusted_value_backtest(
    metrics=metrics,
    income_history=income_history,
    prices=prices[prices["symbol"].isin(eligible_universe)],
    rank_col="ebitda_yield",
)

pe_returns, pe_selections, _ = run_debt_adjusted_value_backtest(
    metrics=metrics,
    income_history=income_history,
    prices=prices[prices["symbol"].isin(eligible_universe)],
    rank_col="earnings_yield",
)

live_dates = strategy_returns.loc[strategy_returns["return"].notna(), "date"]
holdings_per_date = selections.groupby("as_of")["symbol"].nunique() if not selections.empty else pd.Series(dtype=int)

print(f"\nRebalance dates with data: {len(rebalance_dates)}")
print(f"First invested date:       {live_dates.min().date() if not live_dates.empty else 'n/a'}")
print(f"Last rebalance:            {rebalance_dates[-1] if rebalance_dates else 'n/a'}")
if not holdings_per_date.empty:
    print(
        "Holdings per live rebalance: "
        f"min={int(holdings_per_date.min())}  "
        f"median={holdings_per_date.median():.0f}  "
        f"max={int(holdings_per_date.max())}"
    )

print(f"\nLatest holdings ({rebalance_dates[-1] if rebalance_dates else 'n/a'}) — largest weights first:")
if rebalance_dates and not selections.empty:
    latest = selections[selections["as_of"] == rebalance_dates[-1]].copy()
    show = latest.copy()
    for col in ("ebitda_yield", "earnings_yield", "debt_ratio", "weight"):
        if col in show.columns:
            show[col] = show[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "—")
    if "ev_ebitda" in show.columns:
        show["ev_ebitda"] = show["ev_ebitda"].map(lambda x: f"{x:.1f}x" if pd.notna(x) else "—")
    display(show.reset_index(drop=True))

    if not pe_selections.empty:
        p_latest = set(pe_selections.loc[pe_selections["as_of"] == rebalance_dates[-1], "symbol"])
        e_latest = set(latest["symbol"])
        print(
            f"Overlap vs E/P twin this month: {len(e_latest & p_latest)} / {len(e_latest)} "
            f"(EV-only {sorted(e_latest - p_latest)}, E/P-only {sorted(p_latest - e_latest)})"
        )
else:
    print("  (no compliant picks on last rebalance date)")


## Step 7 — Performance vs SPY, S&P 500, and SPUS

Compare the live EBITDA/EV book and the E/P twin against the all-stock and Halal benchmarks on the same live window.


In [ ]:
def benchmark_returns(ticker: str) -> pd.Series:
    px = prices[prices["symbol"] == ticker].copy()
    px = px.sort_values("date").set_index("date")["adj_close"]
    return px.pct_change(fill_method=None).dropna()


def align_to_live(strategy: pd.Series, benches: dict[str, pd.Series]) -> pd.DataFrame:
    """Keep only days the strategy is live, and the same days for every benchmark."""
    panel = pd.DataFrame({"strategy": strategy})
    for name, series in benches.items():
        panel[name] = series
    return panel.dropna(how="any")


fig_dir = NB_DIR / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

bench_series = {name: benchmark_returns(ticker) for name, ticker in BENCHMARKS.items()}
strategy_s = strategy_returns.set_index("date")["return"].rename(STRATEGY_LABEL)
pe_s = pe_returns.set_index("date")["return"].rename(PE_LABEL)

main_panel = align_to_live(strategy_s, bench_series).rename(columns={"strategy": STRATEGY_LABEL})
pe_panel = align_to_live(pe_s, bench_series).rename(columns={"strategy": PE_LABEL})

if main_panel.empty:
    raise RuntimeError("No overlapping live days between the strategy and benchmarks.")

print(
    "Live window (EBITDA/EV): "
    f"{main_panel.index.min().date()} → {main_panel.index.max().date()}  "
    f"({len(main_panel)} trading days)"
)
if not pe_panel.empty:
    print(
        "Live window (E/P twin): "
        f"{pe_panel.index.min().date()} → {pe_panel.index.max().date()}  "
        f"({len(pe_panel)} trading days)"
    )

rows = [calc_performance_stats(main_panel[STRATEGY_LABEL], STRATEGY_LABEL)]
if not pe_panel.empty:
    rows.append(calc_performance_stats(pe_panel[PE_LABEL], PE_LABEL))
for name in BENCHMARKS:
    rows.append(calc_performance_stats(main_panel[name], name))

performance = pd.DataFrame(rows).set_index("Strategy")
formatters = {
    "CAGR": "{:.2%}",
    "Volatility": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}",
    "Max Drawdown": "{:.2%}",
    "Calmar": "{:.2f}",
}
display(performance.style.format(formatters))

# Cumulative growth of $1 over the live window
plot_panel = main_panel.copy()
if not pe_panel.empty:
    plot_panel = plot_panel.join(pe_panel[[PE_LABEL]], how="left")
equity_curves = (1 + plot_panel).cumprod()
equity_curves.plot(figsize=(11, 5), title="Debt-adjusted value (EBITDA/EV) vs Benchmarks (live window)")
plt.ylabel("Growth of $1")
plt.xlabel("")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / "equity-curves.png", dpi=140)
plt.show()

# Rolling 12-month excess return vs SPUS (Halal benchmark)
if "SPUS" in main_panel.columns:
    rolling_excess = (
        (1 + main_panel[STRATEGY_LABEL]).rolling(252).apply(np.prod, raw=True)
        - (1 + main_panel["SPUS"]).rolling(252).apply(np.prod, raw=True)
    )
    rolling_excess.plot(figsize=(11, 3.5), color="tab:green", title="Rolling 12-Month Excess Return vs SPUS")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.ylabel("Excess return")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_dir / "rolling-excess.png", dpi=140)
    plt.show()


## Step 8 — How bumpy was the ride, and in which years?

Drawdowns, calendar-year returns, holdings count, and value diagnostics (cheapness / leverage) on the live book.


In [ ]:
# Drawdowns from running peaks (live window)
equity_live = (1 + main_panel).cumprod()
drawdowns = equity_live / equity_live.cummax() - 1
drawdowns.plot(figsize=(11, 4), title="Drawdowns (live window)")
plt.ylabel("Drawdown")
plt.xlabel("")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / "drawdowns.png", dpi=140)
plt.show()

# Calendar-year returns
yearly = (1 + main_panel).resample("YE").prod() - 1
yearly.index = yearly.index.year
yearly.index.name = "Year"
display(yearly.style.format("{:.2%}").set_caption("Calendar-year returns (live window)"))

yearly.plot(kind="bar", figsize=(11, 4), title="Calendar-year returns")
plt.ylabel("Return")
plt.axhline(0, color="black", linewidth=0.8)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / "calendar-year-returns.png", dpi=140)
plt.show()

# How many names were held each month
if selections.empty:
    print("No holdings log — skip concentration chart.")
else:
    holdings_count = selections.groupby("as_of")["symbol"].nunique().sort_index()
    holdings_count.index = pd.to_datetime(holdings_count.index)
    holdings_count.plot(
        figsize=(11, 3),
        marker="o",
        title="Number of holdings at each rebalance",
    )
    plt.ylabel("Names held")
    plt.xlabel("")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_dir / "holdings-count.png", dpi=140)
    plt.show()
    print(
        f"Holdings: min={int(holdings_count.min())}  "
        f"median={holdings_count.median():.0f}  "
        f"max={int(holdings_count.max())}"
    )

    value_diag = (
        selections.groupby("as_of")
        .agg(
            n=("symbol", "nunique"),
            avg_ebitda_yield=("ebitda_yield", "mean"),
            avg_earnings_yield=("earnings_yield", "mean"),
            med_ev_ebitda=("ev_ebitda", "median"),
            avg_debt_ratio=("debt_ratio", "mean"),
        )
        .sort_index()
    )
    value_diag.index = pd.to_datetime(value_diag.index)
    display(
        value_diag.tail(12).style.format({
            "avg_ebitda_yield": "{:.2%}",
            "avg_earnings_yield": "{:.2%}",
            "med_ev_ebitda": "{:.1f}x",
            "avg_debt_ratio": "{:.2%}",
        }).set_caption("Live-book value & leverage (last 12 rebalances)")
    )
    print(
        f"Sample averages — EBITDA/EV {value_diag['avg_ebitda_yield'].mean():.2%}  ·  "
        f"E/P {value_diag['avg_earnings_yield'].mean():.2%}  ·  "
        f"median EV/EBITDA {value_diag['med_ev_ebitda'].median():.1f}x  ·  "
        f"debt ratio {value_diag['avg_debt_ratio'].mean():.2%}"
    )


## Step 9 — Does cheaper EBITDA/EV actually earn more?

Cap-weight five EBITDA/EV buckets inside the Halal pool. Q1 is cheapest (highest EBITDA/EV); Q5 is most expensive. The live strategy is roughly Q1.


In [ ]:
def cap_weighted_returns(
    weights_by_date: dict[date, pd.Series],
    daily_returns: pd.DataFrame,
) -> pd.Series:
    """Daily returns from a {rebalance_date: symbol-weight Series} map."""
    rebal = sorted(weights_by_date)
    active_w = pd.Series(dtype=float)
    last: date | None = None
    live = False
    rows: list[float] = []

    for dt in daily_returns.index:
        if dt.date() < pd.Timestamp(START).date():
            rows.append(np.nan)
            continue
        prior = [d for d in rebal if pd.Timestamp(d) <= dt]
        if prior and prior[-1] != last:
            active_w = weights_by_date.get(prior[-1], pd.Series(dtype=float))
            last = prior[-1]
            if not active_w.empty:
                live = True
        if not live:
            rows.append(np.nan)
            continue
        held = [s for s in active_w.index if s in daily_returns.columns]
        w = active_w.reindex(held).astype(float)
        w = w[w > 0]
        if w.empty:
            rows.append(0.0)
            continue
        w = w / w.sum()
        day_ret = (daily_returns.loc[dt, w.index] * w).sum(skipna=True)
        rows.append(0.0 if pd.isna(day_ret) else float(day_ret))

    return pd.Series(rows, index=pd.to_datetime(daily_returns.index), name="return")


N_Q = 5
scored_panel = attach_latest_income(metrics, income_history)
screener = AAOIFIScreener(debt_threshold=DEBT_THRESHOLD)
price_panel = (
    prices[prices["symbol"].isin(eligible_universe)]
    .pivot(index="date", columns="symbol", values="adj_close")
    .sort_index()
    .ffill()
)
stock_returns = price_panel.pct_change(fill_method=None)

quantile_w: dict[int, dict[date, pd.Series]] = {q: {} for q in range(1, N_Q + 1)}
for as_of in rebalance_dates:
    ranked = select_debt_adjusted_value(
        scored_panel,
        as_of=as_of,
        rank_col="ebitda_yield",
        keep_quantile=1.0,
        min_holdings=1,
        screener=screener,
    )
    if ranked.empty:
        continue
    ranked = ranked.sort_values("ebitda_yield", ascending=False)
    n = len(ranked)
    if n < N_Q * 2:
        continue
    edges = np.linspace(0, n, N_Q + 1).astype(int)
    for q in range(N_Q):
        chunk = assign_cap_weights(ranked.iloc[edges[q] : edges[q + 1]].copy())
        quantile_w[q + 1][as_of] = chunk.set_index("symbol")["weight"].astype(float)

quantile_returns = pd.DataFrame(
    {
        f"Q{q}": cap_weighted_returns(quantile_w[q], stock_returns)
        for q in range(1, N_Q + 1)
    }
)
quantile_live = quantile_returns.reindex(main_panel.index).dropna(how="any")

if quantile_live.empty:
    print("Not enough names for a clean quintile sort in this window — skip.")
else:
    q_rows = [calc_performance_stats(quantile_live[col], col) for col in quantile_live.columns]
    q_stats = pd.DataFrame(q_rows).set_index("Strategy")
    display(q_stats.style.format(formatters).set_caption(
        "EBITDA/EV quintiles inside the Halal pool  ·  Q1 = cheapest (highest EBITDA/EV), cap-weighted"
    ))

    q_equity = (1 + quantile_live).cumprod()
    q_equity.plot(figsize=(11, 5), title="Growth of $1 by EBITDA/EV quintile (live window)")
    plt.ylabel("Growth of $1")
    plt.xlabel("")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_dir / "quintile-growth.png", dpi=140)
    plt.show()

    if {"Q1", "Q5"}.issubset(quantile_live.columns):
        spread = quantile_live["Q1"] - quantile_live["Q5"]
        spread_equity = (1 + spread).cumprod()
        spread_equity.plot(
            figsize=(11, 3.5),
            color="tab:purple",
            title="Q1 minus Q5 cumulative spread (cheap vs expensive EBITDA/EV)",
        )
        plt.axhline(1.0, color="black", linewidth=0.8)
        plt.ylabel("Growth of $1 on the spread")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(fig_dir / "quintile-spread.png", dpi=140)
        plt.show()
        print(
            f"Q1−Q5 annualized spread: {spread.mean() * 252:.2%}  ·  "
            f"hit rate (Q1 beats Q5 that day): {(quantile_live['Q1'] > quantile_live['Q5']).mean():.1%}"
        )


## Step 10 — Attribution: stock picking or a hidden sector bet?

CAPM alpha/beta vs each benchmark, plus average sector weights of the live book versus the Halal universe.


In [ ]:
def capm_attribution(strategy: pd.Series, benchmark: pd.Series, rf: float = RISK_FREE_RATE) -> dict:
    """Alpha, beta, R², tracking error vs one benchmark (daily, then annualized)."""
    aligned = pd.concat([strategy, benchmark], axis=1, keys=["y", "x"]).dropna()
    rf_d = rf / 252
    ye = aligned["y"] - rf_d
    xe = aligned["x"] - rf_d
    var_x = xe.var()
    beta = float(ye.cov(xe) / var_x) if var_x > 0 else np.nan
    alpha_d = float(ye.mean() - beta * xe.mean()) if pd.notna(beta) else np.nan
    resid = ye - alpha_d - beta * xe
    r2 = 1 - resid.var() / ye.var() if ye.var() > 0 else np.nan
    excess = aligned["y"] - aligned["x"]
    te = float(excess.std() * np.sqrt(252))
    ir = float(excess.mean() * 252 / te) if te > 0 else np.nan
    return {
        "Alpha (ann.)": alpha_d * 252,
        "Beta": beta,
        "R²": r2,
        "Tracking error": te,
        "Information ratio": ir,
    }


attr_rows = []
for bench_name in BENCHMARKS:
    row = capm_attribution(main_panel[STRATEGY_LABEL], main_panel[bench_name])
    row["Benchmark"] = bench_name
    attr_rows.append(row)
attr = pd.DataFrame(attr_rows).set_index("Benchmark")
display(
    attr.style.format({
        "Alpha (ann.)": "{:.2%}",
        "Beta": "{:.2f}",
        "R²": "{:.2f}",
        "Tracking error": "{:.2%}",
        "Information ratio": "{:.2f}",
    }).set_caption("EBITDA/EV value vs each benchmark (live window)")
)

if "activity_labels" not in globals():
    _, _, activity_labels = apply_sector_screen(list(eligible_universe))

universe_w = (
    pd.Series({s: activity_labels.get(s, "Unclassified") for s in eligible_universe})
    .value_counts(normalize=True)
    .rename("Halal universe (equal weight)")
)

if selections.empty:
    print("No holdings log — skip sector mix.")
else:
    month_weights = []
    for as_of, snap in selections.groupby("as_of"):
        w = snap["weight"] if "weight" in snap.columns else 1 / len(snap)
        month_weights.append(
            snap.assign(weight=w)
            .assign(sector=lambda d: d["symbol"].map(activity_labels).fillna("Unclassified"))
            .groupby("sector")["weight"]
            .sum()
        )
    strat_w = pd.concat(month_weights, axis=1).mean(axis=1).rename("EBITDA/EV book (avg. month)")
    sector_mix = pd.concat([strat_w, universe_w], axis=1).fillna(0.0)
    sector_mix["Delta"] = sector_mix["EBITDA/EV book (avg. month)"] - sector_mix["Halal universe (equal weight)"]
    sector_mix = sector_mix.sort_values("EBITDA/EV book (avg. month)", ascending=False)
    display(sector_mix.style.format("{:.1%}").set_caption(
        "Average sector weights  ·  delta = strategy minus Halal universe"
    ))
    sector_mix.drop(columns="Delta").plot(
        kind="barh",
        figsize=(11, max(4, 0.4 * len(sector_mix))),
        title="Sector mix: EBITDA/EV book vs Halal universe",
    )
    plt.xlabel("Average weight")
    plt.tight_layout()
    plt.savefig(fig_dir / "sector-mix.png", dpi=140)
    plt.show()


## Step 11 — Friction: turnover, trading costs, purification

One-way turnover between monthly books, assumed trading-cost drag, and dividend purification drag on names while held. Value traps that fail AAOIFI mid-sample show up as forced exits in the turnover path.


In [ ]:
def one_way_turnover(old: dict[str, float], new: dict[str, float]) -> float:
    """Half L1 distance between two weight books (0 = no trades, 1 = full replace)."""
    if not new:
        return 0.0
    if not old:
        return 1.0
    names = set(old) | set(new)
    return 0.5 * sum(abs(new.get(s, 0.0) - old.get(s, 0.0)) for s in names)


def estimate_purification_drag(
    selection_log: pd.DataFrame,
    price_panel: pd.DataFrame,
    live_index: pd.DatetimeIndex,
    label: str,
) -> tuple[float, pd.DataFrame, int]:
    """Annualized purification drag on dividends received while names were held."""
    if selection_log.empty:
        return np.nan, pd.DataFrame(), 0

    held_symbols = sorted(selection_log["symbol"].unique())
    try:
        purified = hq.purify_dividends(
            held_symbols,
            start=str(live_index.min().date()),
            end=END,
        )
    except Exception as exc:
        print(f"[{label}] Purification lookup failed ({exc}).")
        return np.nan, pd.DataFrame(), 0

    if purified is None or purified.empty:
        print(f"[{label}] No purification rows — report as a data gap, not 0% drag.")
        return np.nan, pd.DataFrame(), 0

    held_flags = []
    for as_of, snap in selection_log.groupby("as_of"):
        held_flags.append(pd.DataFrame({"as_of": pd.Timestamp(as_of), "symbol": snap["symbol"].tolist()}))
    held_log = pd.concat(held_flags, ignore_index=True)

    purified = purified.copy()
    purified["ex_date"] = pd.to_datetime(purified["ex_date"])
    px = price_panel.copy()
    px.index = pd.to_datetime(px.index)

    weight_lookup = {}
    for as_of, snap in selection_log.groupby("as_of"):
        if "weight" in snap.columns:
            weight_lookup[pd.Timestamp(as_of)] = snap.set_index("symbol")["weight"].astype(float).to_dict()
        else:
            weight_lookup[pd.Timestamp(as_of)] = {s: 1.0 / len(snap) for s in snap["symbol"]}

    drag_rows = []
    for _, div in purified.iterrows():
        symbol = div["symbol"]
        ex = div["ex_date"]
        prior_rebal = held_log.loc[held_log["as_of"] <= ex]
        if prior_rebal.empty:
            continue
        last_asof = prior_rebal["as_of"].max()
        book_w = weight_lookup.get(last_asof, {})
        if symbol not in book_w:
            continue
        if symbol not in px.columns:
            continue
        hist = px.loc[:ex, symbol].dropna()
        if hist.empty or hist.iloc[-1] <= 0:
            continue
        amount = div.get("purification_amount")
        if pd.isna(amount):
            continue
        drag_rows.append(float(amount) / float(hist.iloc[-1]) * float(book_w[symbol]))

    years = len(live_index) / 252
    purify_drag = (sum(drag_rows) / years) if years > 0 else np.nan
    return purify_drag, purified, len(drag_rows)


if selections.empty:
    raise RuntimeError("No holdings log — re-run the backtest cell first.")

if "price_panel" not in globals():
    price_panel = (
        prices[prices["symbol"].isin(eligible_universe)]
        .pivot(index="date", columns="symbol", values="adj_close")
        .sort_index()
        .ffill()
    )

rebal_order = sorted(selections["as_of"].unique())
turnover_rows = []
prev: dict[str, float] = {}
for as_of in rebal_order:
    snap = selections.loc[selections["as_of"] == as_of]
    if "weight" in snap.columns:
        curr = snap.set_index("symbol")["weight"].astype(float).to_dict()
    else:
        curr = {s: 1.0 / len(snap) for s in snap["symbol"]}
    turnover_rows.append({"as_of": pd.Timestamp(as_of), "one_way": one_way_turnover(prev, curr), "n": len(curr)})
    prev = curr

turnover = pd.DataFrame(turnover_rows).set_index("as_of")
months = max(len(turnover) - 1, 1)
annual_turnover = turnover["one_way"].iloc[1:].sum() * (12 / months) if len(turnover) > 1 else np.nan
cost_drag_ann = annual_turnover * COST_BPS / 10_000

cost_on_day = pd.Series(0.0, index=main_panel.index)
for as_of, row in turnover.iterrows():
    hit = main_panel.index[main_panel.index >= as_of]
    if len(hit):
        cost_on_day.loc[hit[0]] += float(row["one_way"]) * COST_BPS / 10_000

net_returns = main_panel[STRATEGY_LABEL] - cost_on_day
net_stats = calc_performance_stats(net_returns, f"{STRATEGY_LABEL} net of {COST_BPS} bps")
gross_cagr = calc_performance_stats(main_panel[STRATEGY_LABEL], STRATEGY_LABEL)["CAGR"]

print("Fetching dividend purification for EBITDA/EV book …")
ev_drag, purified_ev, n_ev_events = estimate_purification_drag(
    selections, price_panel, main_panel.index, STRATEGY_LABEL
)
print("Fetching dividend purification for E/P twin …")
pe_drag, purified_pe, n_pe_events = estimate_purification_drag(
    pe_selections, price_panel, pe_panel.index if not pe_panel.empty else main_panel.index, PE_LABEL
)

friction = pd.DataFrame(
    {
        "Item": [
            "Rebalances with holdings",
            "Average names held",
            "Annualized one-way turnover",
            "Assumed cost per one-way trade",
            "Estimated annual trading-cost drag",
            "CAGR gross of trading costs",
            "CAGR net of trading costs",
            "Purification drag (EBITDA/EV book)",
            "Purification drag (E/P twin)",
            "Dividend events while held (EBITDA/EV)",
        ],
        "Value": [
            f"{len(turnover)}",
            f"{turnover['n'].mean():.1f}",
            f"{annual_turnover:.1%}",
            f"{COST_BPS} bps",
            f"{cost_drag_ann:.2%}",
            f"{gross_cagr:.2%}",
            f"{net_stats['CAGR']:.2%}",
            f"{ev_drag:.2%}" if pd.notna(ev_drag) else "data gap",
            f"{pe_drag:.2%}" if pd.notna(pe_drag) else "data gap",
            f"{n_ev_events}",
        ],
    }
)
display(friction.style.hide(axis="index"))

turnover["one_way"].plot(figsize=(11, 3), marker="o", title="One-way turnover at each rebalance")
plt.ylabel("Turnover")
plt.xlabel("")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / "turnover.png", dpi=140)
plt.show()

if not purified_ev.empty:
    display(
        purified_ev[["symbol", "ex_date", "dividend", "impure_ratio", "purification_amount"]]
        .head(12)
        .style.format({"dividend": "{:.3f}", "impure_ratio": "{:.2%}", "purification_amount": "{:.4f}"}, na_rep="—")
        .set_caption("Sample purified dividends (EBITDA/EV book holdings)")
    )


### Notes on methodology

- **Point-in-time:** `halalquant.get_financial_metrics(..., freq="ME")` uses SEC `filed_date` (or report date + 90 days for non-US filers) so rebalance decisions never use future filings.
- **Income source:** EBITDA and net income come from yfinance annual income statements with the same 90-day publication lag. If Yahoo lacks an EBITDA line, the notebook falls back to operating income + depreciation & amortization when both exist. Yahoo typically only stores a few annual periods, so the book may go live later than `START`.
- **Enterprise value:** \(\mathrm{EV} = \mathrm{market\_cap} + \mathrm{total\_debt} - \mathrm{cash\_and\_equiv}\) from the AAOIFI metrics snapshot on each rebalance date.
- **Selection:** Halal AAOIFI filter → positive EBITDA, \(\mathrm{EV} > 0\), \(\mathrm{EBITDA/EV} > 0\) → top `KEEP_QUANTILE` by **EBITDA/EV**, cap-weighted. The E/P twin only changes the ranking column (and requires positive net income).
- **Live-window stats:** Days before the first real holdings are missing, not 0%. Benchmarks are sliced to that same window.
- **Quintiles:** Q1–Q5 split the Halal pool by EBITDA/EV. Q1 is cheapest (near the live book).
- **Friction:** Turnover is half the L1 weight change between monthly books. Trading-cost drag applies `COST_BPS` on each one-way turn. Purification drag is reported for both books.
- **Benchmarks:** SPY (tradable S&P 500 ETF), `^GSPC` (S&P 500 index), SPUS (Halal large-cap ETF).
- **Relation to paper 01:** Paper 01 dropped an FCF/EV *cheapness* rule after it lost to SPUS in 2023–2024. This notebook re-tests debt-adjusted cheapness with EBITDA/EV and an explicit E/P twin inside AAOIFI screens.
- **Runtime:** `FAST_MODE = False` runs the full S&P 500 (~15–45 min depending on network). Figures save to `Research/papers/08-debt-adj-value/figures/`.
- **White-paper map:** Step 7 → section 3; Steps 8–9 → section 3 extras; Step 10 → section 4; Step 11 → section 5. Hypothesis and strategy rules (sections 1–2) are written in the paper, not computed here.
- **Survivorship:** the S&P 500 list is current; historical membership is not reconstructed. Treat results as exploratory research, not a live mandate.


## Summary

| Component | Rule |
| --- | --- |
| Universe | S&P 500 → Halal sector screen → AAOIFI financial ratios |
| Signal | EBITDA / Enterprise Value (high = cheap) |
| EV definition | Market cap + total debt − cash |
| Portfolio | Top quintile by EBITDA/EV, cap-weighted, monthly rebalance |
| Control twin | Same Halal filters, ranked by **earnings yield** (E/P) |
| Window | 2020-01-01 → 2025-12-31 |
| Benchmarks | SPY, S&P 500 (`^GSPC`), SPUS |
| Focus metric | Trap avoidance vs E/P twin; factor monotonicity; sector tilt |

Re-run with `FAST_MODE = False` for the paper figures. Follow up if EBITDA/EV beats or matches SPUS *and* looks cleaner than the E/P twin on quintiles / forced AAOIFI exits; revise (e.g. add an FCF or ROIC overlay) if cheapness alone fails again; kill if the edge is only a sector bet.
